In [5]:
import ansys.aedt.core
import os
import tempfile
import time
from ansys.aedt.core import Maxwell3d
from ansys.aedt.core import Maxwell2d

AEDT_VERSION = "2025.1"
NUM_CORES = 8
NG_MODE = False  # Open AEDT UI when it is launched.

In [6]:
# aedtPath=r'F:\KDH\Thesis\JEET\e10\refModel\e10_UserRemesh_ANSYSEM_2D.aedt'
# aedtPath=r'F:\KDH\ANSYS_TRC\ALH\Maxw_GS_EN_ILT\M04\Workshop_files\WS4.1\Maxw_GS_2DmagTrans1.aedt'
aedtPath=r'C:\ANSYS_Motor-CAD\2025_1_1\Tutorials\Ansys_Maxwell_Lab\e10_example\MaxwellLabTutorial.aedt'

In [9]:
# AEDT 파일 Lock 체크 및 안전한 파일 열기
import os
import time
import psutil
from pathlib import Path

def check_file_lock(file_path, timeout=30):
    """
    AEDT 파일이 잠겨있는지 확인하고 안전하게 열 수 있는 상태인지 체크
    
    Parameters:
    - file_path: 체크할 AEDT 파일 경로
    - timeout: 최대 대기 시간 (초)
    
    Returns:
    - True: 파일을 안전하게 열 수 있음
    - False: 파일이 잠겨있음
    """
    file_path = Path(file_path)
    
    # 파일 존재 여부 확인
    if not file_path.exists():
        print(f"❌ 파일이 존재하지 않습니다: {file_path}")
        return False
    
    # Lock 파일들 확인 (.lock, .lck, .tmp 등)
    lock_extensions = ['.lock', '.lck', '.tmp']
    lock_files = []
    
    for ext in lock_extensions:
        lock_file = file_path.with_suffix(file_path.suffix + ext)
        if lock_file.exists():
            lock_files.append(lock_file)
    
    # AEDT 관련 프로세스가 파일을 사용 중인지 확인
    def is_file_in_use():
        try:
            # 파일을 독점 모드로 열어보기
            with open(file_path, 'r+b') as f:
                pass
            return False
        except (IOError, OSError, PermissionError):
            return True
    
    # AEDT 관련 프로세스 확인
    def check_aedt_processes():
        aedt_processes = []
        for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
            try:
                name = proc.info['name'].lower()
                if any(aedt_name in name for aedt_name in ['ansysedt', 'maxwell', 'hfss', 'q3d']):
                    cmdline = proc.info.get('cmdline', [])
                    if cmdline and str(file_path) in ' '.join(cmdline):
                        aedt_processes.append(proc.info)
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                continue
        return aedt_processes
    
    print(f"📁 파일 확인: {file_path.name}")
    
    # Lock 파일 존재 확인
    if lock_files:
        print(f"⚠️  Lock 파일 발견: {[f.name for f in lock_files]}")
    
    # AEDT 프로세스 확인
    aedt_procs = check_aedt_processes()
    if aedt_procs:
        print(f"⚠️  AEDT 프로세스가 파일을 사용 중: {len(aedt_procs)}개 프로세스")
    
    # 파일 사용 상태 확인
    start_time = time.time()
    while time.time() - start_time < timeout:
        if not is_file_in_use() and not lock_files and not aedt_procs:
            print("✅ 파일을 안전하게 열 수 있습니다.")
            return True
        
        if time.time() - start_time < timeout:
            print(f"⏳ 파일이 사용 중입니다. 대기 중... ({int(time.time() - start_time)}/{timeout}초)")
            time.sleep(2)
            
            # 상태 재확인
            lock_files = [f for f in lock_files if f.exists()]
            aedt_procs = check_aedt_processes()
    
    print(f"❌ {timeout}초 대기 후에도 파일이 잠겨있습니다.")
    return False

def safe_open_aedt(file_path, max_retries=3):
    """
    AEDT 파일을 안전하게 여는 함수
    
    Parameters:
    - file_path: AEDT 파일 경로
    - max_retries: 최대 재시도 횟수
    
    Returns:
    - Maxwell2d 객체 또는 None
    """
    for attempt in range(max_retries):
        print(f"\n=== 시도 {attempt + 1}/{max_retries} ===")
        
        if check_file_lock(file_path):
            try:
                print("🔄 Maxwell2d 인스턴스 생성 중...")
                m2d = Maxwell2d(
                    project=file_path,
                    version=AEDT_VERSION,
                    new_desktop=False,
                    non_graphical=NG_MODE,
                )
                print("✅ Maxwell2d 인스턴스 생성 성공!")
                return m2d
                
            except Exception as e:
                print(f"❌ Maxwell2d 생성 실패: {e}")
                if attempt < max_retries - 1:
                    print("⏳ 5초 후 재시도...")
                    time.sleep(5)
        else:
            if attempt < max_retries - 1:
                print("⏳ 10초 후 재시도...")
                time.sleep(10)
    
    print("❌ 모든 시도가 실패했습니다.")
    return None

print("파일 Lock 체크 함수들이 정의되었습니다.")

파일 Lock 체크 함수들이 정의되었습니다.


In [8]:
from ansys.aedt.core import Maxwell2d

print("=== AEDT 파일 안전하게 열기 ===")

# 새로운 안전한 방식
m2d = safe_open_aedt(aedtPath, max_retries=3)

if m2d is not None:
    print(f"\n✅ 성공적으로 열렸습니다!")
    print(f"  프로젝트: {m2d.project_name}")
    print(f"  디자인: {m2d.design_name}")
    print(f"  솔루션 타입: {m2d.solution_type}")
    print(f"  버전: {m2d.aedt_version_id}")
else:
    print("\n❌ 파일을 열 수 없습니다. 다음을 확인해주세요:")
    print("  1. 다른 AEDT 인스턴스에서 파일이 열려있는지 확인")
    print("  2. 파일 경로가 올바른지 확인")
    print("  3. 파일 권한 확인")
    print("  4. AEDT 버전 호환성 확인")

=== AEDT 파일 안전하게 열기 ===

=== 시도 1/3 ===
📁 파일 확인: MaxwellLabTutorial.aedt
⚠️  Lock 파일 발견: ['MaxwellLabTutorial.aedt.lock']
⏳ 파일이 사용 중입니다. 대기 중... (0/30초)
⏳ 파일이 사용 중입니다. 대기 중... (0/30초)
⏳ 파일이 사용 중입니다. 대기 중... (6/30초)
⏳ 파일이 사용 중입니다. 대기 중... (6/30초)
⏳ 파일이 사용 중입니다. 대기 중... (12/30초)
⏳ 파일이 사용 중입니다. 대기 중... (12/30초)
⏳ 파일이 사용 중입니다. 대기 중... (19/30초)
⏳ 파일이 사용 중입니다. 대기 중... (19/30초)
⏳ 파일이 사용 중입니다. 대기 중... (25/30초)
⏳ 파일이 사용 중입니다. 대기 중... (25/30초)
❌ 30초 대기 후에도 파일이 잠겨있습니다.
⏳ 10초 후 재시도...
❌ 30초 대기 후에도 파일이 잠겨있습니다.
⏳ 10초 후 재시도...

=== 시도 2/3 ===
📁 파일 확인: MaxwellLabTutorial.aedt

=== 시도 2/3 ===
📁 파일 확인: MaxwellLabTutorial.aedt
✅ 파일을 안전하게 열 수 있습니다.
🔄 Maxwell2d 인스턴스 생성 중...
PyAEDT INFO: Parsing C:\ANSYS_Motor-CAD\2025_1_1\Tutorials\Ansys_Maxwell_Lab\e10_example\MaxwellLabTutorial.aedt.
PyAEDT INFO: Python version 3.8.10 (tags/v3.8.10:3d8993a, May  3 2021, 11:48:03) [MSC v.1928 64 bit (AMD64)]
PyAEDT INFO: PyAEDT version 0.13.0.
✅ 파일을 안전하게 열 수 있습니다.
🔄 Maxwell2d 인스턴스 생성 중...
PyAEDT INFO: Parsing C:\ANSYS_Mot

## PJT Info

### getter all

In [ ]:
dirofItemsM2d = vars(m2d)
for attr in dirofItemsM2d:
    try:
        typeofDirListM2d[attr] = type(getattr(m2d, attr))
    except Exception as e:
        typeofDirListM2d[attr] = str(e)

### dev

In [ ]:
typeofItems={}
for k, v in vars(m2d).items():
    typeofItems[k] = type(v)
    print(f"{k}: {type(v)}")

varsofItems={}
for k, v in vars(m2d):
    varsofItems[k] = type(v)
    print(f"{k}: {type(v)}")

In [ ]:
# 디자인 이름 목록 추출
design_names = m2d.design_list
print("Designs in the project:", design_names)

## geometry objects parsing

In [ ]:
import json

# AEDT 프로젝트 열기

# 객체 리스트 가져오기
ob3dlist = m2d.modeler.object_list


In [ ]:
def parse_objects_with_instance(obj_list):
    parsed_objects = []
    for obj in obj_list:
        obj_info = {
            "name": getattr(obj, "name", None),
            "material": getattr(obj, "material_name", None),
            "model": getattr(obj, "model", None),
            "volume": getattr(obj, "volume", None),
            "mass": getattr(obj, "mass", None),
            "color": getattr(obj, "color", None),
            "transparency": getattr(obj, "transparency", None),
            "bounding_box": getattr(obj, "bounding_box", None),
            "solve_inside": getattr(obj, "solve_inside", None),
            "coordinate_system": getattr(obj, "part_coordinate_system", None),
            "faces": [],
            "edges": [],
            "vertices": [],
            "object": obj  # 객체 자체 저장
        }

        # Faces
        if hasattr(obj, "faces"):
            for face in obj.faces:
                face_info = {
                    "id": getattr(face, "id", None),
                    "area": getattr(face, "area", None),
                    "center": getattr(face, "center", None),
                    "vertices": []
                }
                if hasattr(face, "vertices"):
                    for vertex in face.vertices:
                        face_info["vertices"].append({
                            "id": getattr(vertex, "id", None),
                            "position": getattr(vertex, "position", None)
                        })
                obj_info["faces"].append(face_info)

        # Edges
        if hasattr(obj, "edges"):
            for edge in obj.edges:
                edge_info = {
                    "id": getattr(edge, "id", None),
                    "length": getattr(edge, "length", None)
                }
                obj_info["edges"].append(edge_info)

        # Vertices (global)
        if hasattr(obj, "vertices"):
            for vertex in obj.vertices:
                obj_info["vertices"].append({
                    "id": getattr(vertex, "id", None),
                    "position": getattr(vertex, "position", None)
                })

        parsed_objects.append(obj_info)
    return parsed_objects

# 사용 예시
parsed_objects = parse_objects_with_instance(ob3dlist)


In [ ]:
len(parsed_objects)

In [ ]:
parsed_structure = parse_object(parsed_objects[49]['object'], max_depth=2)


In [ ]:
import json

def safe_str(obj):
    """객체를 JSON 직렬화 가능하게 문자열로 변환 (예외 대비)"""
    try:
        json.dumps(obj)
        return obj
    except:
        return str(obj)

def parse_object(obj, depth=0, max_depth=3):
    """객체 속성을 재귀적으로 파싱해서 dict로 변환"""
    if depth > max_depth:
        return f"<Max depth {max_depth} reached>"

    result = {}
    for attr in dir(obj):
        if attr.startswith("_"):
            continue
        try:
            value = getattr(obj, attr)
            if callable(value):
                continue

            # 기본 타입이면 그대로 저장
            if isinstance(value, (int, float, str, bool, list, dict, type(None))):
                result[attr] = safe_str(value)
            # 재귀적 구조로 들어갈 수 있는 클래스형 속성
            # elif hasattr(value, "__dict__") or isinstance(value, object):
            elif  isinstance(value, object):
                result[attr] = parse_object(value, depth + 1, max_depth)
            else:
                result[attr] = safe_str(value)

        except Exception as e:
            result[attr] = f"<Error: {str(e)}>"

    return result



# JSON 출력
# print(json_string)


## Excitation

In [ ]:
excitation_list=m2d.excitations

In [ ]:
for name, obj in m2d.boundaries.items():
    if name in excitation_list:
        print(f"--- {name} ---")
        print(obj.props)  # 속성 딕셔너리 출력


### dev

In [ ]:
for name in m2d.excitations:
    if name in m2d.boundaries:
        bdry = m2d.boundaries[name]
        print(f"[{name}] Type: {bdry.type}")
        for key, value in bdry.props.items():
            print(f"    {key}: {value}")


In [ ]:
json_string = json.dumps(parsed_structure, indent=2, ensure_ascii=False)
print(json_string)


# 구조화된 클래스 데이터 직렬화 방법들

사용자 정의 클래스를 계층 구조를 유지하면서 기본 Python 데이터 타입이나 파일로 변환하는 방법들

In [10]:
# 1. 가장 일반적인 방법들
import json
import pickle
import yaml  # pip install pyyaml
from dataclasses import dataclass, asdict
import pandas as pd

print("=== 주요 직렬화 방법들 ===")

# 1) __dict__ 속성 사용 (기본 방법)
def to_dict_basic(obj):
    """기본적인 __dict__ 사용"""
    if hasattr(obj, '__dict__'):
        return obj.__dict__
    return obj

# 2) vars() 함수 사용 (동일한 결과)
def to_dict_vars(obj):
    """vars() 함수 사용"""
    try:
        return vars(obj)
    except TypeError:
        return obj

# 3) dataclass의 asdict() 사용 (dataclass용)
@dataclass
class ExampleDataClass:
    name: str
    value: int
    nested: dict = None

example_dc = ExampleDataClass("test", 42, {"sub": "value"})
print("dataclass asdict():", asdict(example_dc))

# 4) 재귀적 변환 함수 (복잡한 객체용)
def to_dict_recursive(obj, max_depth=5, current_depth=0):
    """재귀적으로 객체를 dict로 변환"""
    if current_depth >= max_depth:
        return str(obj)
    
    if hasattr(obj, '__dict__'):
        result = {}
        for key, value in obj.__dict__.items():
            if isinstance(value, (str, int, float, bool, type(None))):
                result[key] = value
            elif isinstance(value, (list, tuple)):
                result[key] = [to_dict_recursive(item, max_depth, current_depth + 1) for item in value]
            elif isinstance(value, dict):
                result[key] = {k: to_dict_recursive(v, max_depth, current_depth + 1) for k, v in value.items()}
            else:
                result[key] = to_dict_recursive(value, max_depth, current_depth + 1)
        return result
    elif isinstance(obj, (list, tuple)):
        return [to_dict_recursive(item, max_depth, current_depth + 1) for item in obj]
    elif isinstance(obj, dict):
        return {k: to_dict_recursive(v, max_depth, current_depth + 1) for k, v in obj.items()}
    else:
        return str(obj)

print("재귀적 변환 함수 정의 완료")

=== 주요 직렬화 방법들 ===
dataclass asdict(): {'name': 'test', 'value': 42, 'nested': {'sub': 'value'}}
재귀적 변환 함수 정의 완료


In [ ]:
# 2. 파일 저장 방법들
print("\n=== 파일 저장 방법들 ===")

def save_to_json(obj, filename, use_custom_converter=True):
    """JSON 파일로 저장"""
    if use_custom_converter:
        data = to_dict_recursive(obj)
    else:
        data = obj
    
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False, default=str)
    print(f"✓ JSON 저장: {filename}")

def save_to_pickle(obj, filename):
    """Pickle 파일로 저장 (객체 완전 보존)"""
    with open(filename, 'wb') as f:
        pickle.dump(obj, f)
    print(f"✓ Pickle 저장: {filename}")

def save_to_yaml(obj, filename):
    """YAML 파일로 저장"""
    data = to_dict_recursive(obj)
    with open(filename, 'w', encoding='utf-8') as f:
        yaml.dump(data, f, default_flow_style=False, allow_unicode=True)
    print(f"✓ YAML 저장: {filename}")

def save_to_csv_flat(obj, filename):
    """평면화된 CSV로 저장 (pandas 사용)"""
    data = to_dict_recursive(obj)
    
    # 중첩된 딕셔너리를 평면화
    def flatten_dict(d, parent_key='', sep='_'):
        items = []
        for k, v in d.items():
            new_key = f"{parent_key}{sep}{k}" if parent_key else k
            if isinstance(v, dict):
                items.extend(flatten_dict(v, new_key, sep=sep).items())
            elif isinstance(v, list):
                for i, item in enumerate(v):
                    if isinstance(item, dict):
                        items.extend(flatten_dict(item, f"{new_key}_{i}", sep=sep).items())
                    else:
                        items.append((f"{new_key}_{i}", item))
            else:
                items.append((new_key, v))
        return dict(items)
    
    flat_data = flatten_dict(data)
    df = pd.DataFrame([flat_data])
    df.to_csv(filename, index=False, encoding='utf-8-sig')
    print(f"✓ CSV 저장: {filename}")

print("파일 저장 함수들 정의 완료")

In [11]:
# 3. AEDT 객체에 특화된 직렬화 함수
print("\n=== AEDT 전용 직렬화 함수 ===")

def serialize_aedt_object(aedt_obj, include_geometry=True, include_boundaries=True, max_depth=3):
    """AEDT 객체를 구조화된 딕셔너리로 변환"""
    result = {
        "project_info": {
            "project_name": getattr(aedt_obj, 'project_name', None),
            "design_name": getattr(aedt_obj, 'design_name', None),
            "design_type": getattr(aedt_obj, 'design_type', None),
            "solution_type": getattr(aedt_obj, 'solution_type', None),
            "version": getattr(aedt_obj, 'aedt_version_id', None),
        },
        "settings": {
            "model_units": getattr(aedt_obj.modeler, 'model_units', None) if hasattr(aedt_obj, 'modeler') else None,
            "working_directory": getattr(aedt_obj, 'working_directory', None),
        }
    }
    
    if include_geometry and hasattr(aedt_obj, 'modeler'):
        try:
            result["geometry"] = {
                "object_names": getattr(aedt_obj.modeler, 'object_names', []),
                "object_count": len(getattr(aedt_obj.modeler, 'object_names', [])),
                "objects_detail": []
            }
            
            # 각 객체의 상세 정보
            for obj in getattr(aedt_obj.modeler, 'object_list', []):
                obj_detail = {
                    "name": getattr(obj, 'name', None),
                    "material": getattr(obj, 'material_name', None),
                    "volume": getattr(obj, 'volume', None),
                    "mass": getattr(obj, 'mass', None),
                    "bounding_box": getattr(obj, 'bounding_box', None),
                    "color": getattr(obj, 'color', None),
                }
                result["geometry"]["objects_detail"].append(obj_detail)
        except Exception as e:
            result["geometry"] = {"error": str(e)}
    
    if include_boundaries and hasattr(aedt_obj, 'boundaries'):
        try:
            result["boundaries"] = {}
            for name, boundary in aedt_obj.boundaries.items():
                result["boundaries"][name] = {
                    "type": getattr(boundary, 'type', None),
                    "props": getattr(boundary, 'props', {}),
                }
        except Exception as e:
            result["boundaries"] = {"error": str(e)}
    
    # Excitations
    if hasattr(aedt_obj, 'excitations'):
        try:
            result["excitations"] = list(getattr(aedt_obj, 'excitations', []))
        except Exception as e:
            result["excitations"] = {"error": str(e)}
    
    return result

# m2d 객체에 적용 예제
print("AEDT 객체 직렬화 함수 정의 완료")
print("사용 예: serialized_m2d = serialize_aedt_object(m2d)")


=== AEDT 전용 직렬화 함수 ===
AEDT 객체 직렬화 함수 정의 완료
사용 예: serialized_m2d = serialize_aedt_object(m2d)


In [12]:
# 4. 실제 실행 - m2d 객체를 파일로 내보내기
print("\n=== m2d 객체 직렬화 및 파일 저장 ===")

# m2d 객체를 직렬화
try:
    serialized_m2d = serialize_aedt_object(m2d)
    print("✓ m2d 객체 직렬화 성공")
    
    # 구조 확인
    print(f"프로젝트명: {serialized_m2d['project_info']['project_name']}")
    print(f"디자인명: {serialized_m2d['project_info']['design_name']}")
    print(f"객체 개수: {serialized_m2d['geometry']['object_count']}")
    print(f"경계조건 개수: {len(serialized_m2d['boundaries'])}")
    print(f"여기조건 개수: {len(serialized_m2d['excitations'])}")
    
except Exception as e:
    print(f"직렬화 실패: {e}")
    serialized_m2d = None

# 파일로 저장
if serialized_m2d:
    try:
        # JSON으로 저장
        save_to_json(serialized_m2d, "m2d_maxwell_data.json", use_custom_converter=False)
        
        # YAML로도 저장
        save_to_yaml(serialized_m2d, "m2d_maxwell_data.yaml")
        
        # Pickle로도 저장 (완전한 객체 보존)
        save_to_pickle(serialized_m2d, "m2d_maxwell_data.pkl")
        
        # 평면화된 CSV로 저장
        save_to_csv_flat(serialized_m2d, "m2d_maxwell_data.csv")
        
        print("\n✓ 모든 형식으로 파일 저장 완료!")
        print("저장된 파일들:")
        print("  - m2d_maxwell_data.json (JSON 형식)")
        print("  - m2d_maxwell_data.yaml (YAML 형식)")
        print("  - m2d_maxwell_data.pkl (Pickle 형식)")
        print("  - m2d_maxwell_data.csv (평면화된 CSV)")
        
    except Exception as e:
        print(f"파일 저장 실패: {e}")


=== m2d 객체 직렬화 및 파일 저장 ===
PyAEDT INFO: Modeler2D class has been initialized!
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 1sec
PyAEDT INFO: Modeler2D class has been initialized!
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 1sec
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing objects from Data Model
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing objects from Data Model
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
✓ m2d 객체 직렬화 성공
프로젝트명: MaxwellLabTutorial
디자인명: Motor-CAD_tutorial
객체 개수: 51
경계조건 개수: 1
여기조건 개수: 43
파일 저장 실패: name 'save_to_json' is not defined
✓ m2d 객체 직렬화 성공
프로젝트명: MaxwellLabTutorial
디자인명: Motor-CAD_tutorial
객체 개수: 51
경계조건 개수: